# SCL 模型训练

修复说明：SCL 损失显式使用 FP32；AMP 溢出跳过参数更新时，学习率调度器同步跳过。下方保留的输出来自修复前运行，需要重新运行全部单元格得到新结果。


In [ ]:
import math
import random
import subprocess
import sys
import json
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/prepared_data")
OUTPUT_DIR = Path("/content/drive/MyDrive/contrastive_outputs")


## 1. 环境与依赖


In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers==4.52.4", "sentencepiece>=0.2", "safetensors>=0.4"])
# 保留 Colab 预装的 GPU 版 PyTorch，不替换为 CPU 版。


## 2. 数据路径


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 3. 训练配置


In [ ]:
config = {
    "data_run": str(DATA_DIR),
    "split": "grouped_multilingual",
    "model_name": "FacebookAI/xlm-roberta-base",
    "task": "C",  # A、B、C；每次独立训练一个子任务
    "max_length": 512,
    "dropout": 0.1,
    "projection_dim": 128,
    "epochs": 8,
    "batch_size": 8,
    "eval_batch_size": 8,
    "gradient_accumulation": 2,
    "learning_rate": 2e-5,
    "weight_decay": 0.03,
    "warmup_ratio": 0.1,
    "seed": 42,
    "device": "cuda",
    "contrastive_weight": 0.1,
    "temperature": 0.1,
    "identity_threshold": 0.5,  # 仅用于 C 的身份预测
    "mixed_precision": True,
}
RUN_NAME = f"{config['task']}_scl_seed{config['seed']}"

## 4. 模型与训练函数


### 4.1 导入与标签映射


In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

IDENTITIES = ['l', 'g', 'b', 't', 'q', 'i', 'a', 'nb', 'lgbtqia+']



### 4.2 数据读取与输入编码


In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

In [ ]:
def load_split(data_run, split_name):
    root = Path(data_run)
    records = {}
    for path in sorted((root / 'processed').glob('*.jsonl')):
        for line in path.read_text(encoding='utf-8').splitlines():
            row = json.loads(line)
            records[row['id']] = row
    split = read_json(root / 'splits' / f'{split_name}.json')
    train = [records[i] for i in split['train_ids']]
    val = [records[i] for i in split['validation_ids']]
    return train, val


In [ ]:
def encode_record(row, tokenizer, config):
    text = f"[COMMENT] {row['comment']} [TITLE] {row['title']} [DESCRIPTION] {row['description']}"
    return tokenizer(text, truncation=True, max_length=config['max_length'])


In [ ]:
class TextDataset(Dataset):
    def __init__(self, rows, tokenizer, config):
        self.rows = rows
        self.inputs = [encode_record(row, tokenizer, config) for row in rows]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return {'inputs': self.inputs[index], 'row': self.rows[index]}


In [ ]:
class Collator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, items):
        rows = [item['row'] for item in items]
        batch = {
            'inputs': self.tokenizer.pad([item['inputs'] for item in items], padding=True, return_tensors='pt'),
            'rows': rows,
        }
        for field, dtype in [('stereotype_label', torch.long), ('hate_label', torch.long),
                             ('scope_label', torch.long), ('identity_labels', torch.float32),
                             ('target_loss_mask', torch.bool)]:
            batch[field] = torch.tensor([row[field] for row in rows], dtype=dtype)
        return batch


### 4.3 预测解码与评估指标


In [ ]:
def classification_metrics(gold, pred, labels):
    scores = []
    for label in labels:
        tp = sum(g == label and p == label for g, p in zip(gold, pred))
        fp = sum(g != label and p == label for g, p in zip(gold, pred))
        fn = sum(g == label and p != label for g, p in zip(gold, pred))
        denominator = 2*tp+fp+fn
        scores.append({'label': label, 'support': sum(g == label for g in gold),
                       'precision': tp/(tp+fp) if tp+fp else 0.,
                       'recall': tp/(tp+fn) if tp+fn else 0.,
                       'f1': 2*tp/denominator if denominator else 0.})
    return {'n': len(gold), 'accuracy': sum(g == p for g, p in zip(gold, pred))/len(gold) if gold else None,
            'macro_f1': sum(s['f1'] for s in scores)/len(labels), 'per_class': scores}


In [ ]:
def decode(outputs, task, threshold=.5):
    if task == 'A':
        return [{'stereotype': 'yes' if label else 'no'} for label in outputs['a'].argmax(-1).tolist()]
    if task == 'B':
        return [{'hate_pred': label} for label in outputs['b'].argmax(-1).tolist()]
    scopes = outputs['scope'].argmax(-1).tolist()
    probabilities = outputs['identity'].sigmoid().tolist()
    predictions = []
    for scope, values in zip(scopes, probabilities):
        identities = [int(value >= threshold) for value in values]
        if not any(identities):
            identities[max(range(len(values)), key=values.__getitem__)] = 1
        target = ('group' if scope == 0 else 'individual') + '_' + ','.join(
            name for name, active in zip(IDENTITIES, identities) if active)
        predictions.append({'scope_pred': scope, 'identity_pred': identities, 'target': target})
    return predictions


In [ ]:
def evaluate_metrics(rows, predictions, task):
    if task == 'A':
        return classification_metrics([r['stereotype_label'] for r in rows],
                                      [int(p['stereotype'] == 'yes') for p in predictions], [0, 1])
    if task == 'B':
        return classification_metrics([r['hate_label'] for r in rows],
                                      [p['hate_pred'] for p in predictions], [0, 1, 2])
    scope = classification_metrics([r['scope_label'] for r in rows],
                                   [p['scope_pred'] for p in predictions], [0, 1])
    identities = []
    for i, name in enumerate(IDENTITIES):
        score = classification_metrics([r['identity_labels'][i] for r in rows],
                                       [p['identity_pred'][i] for p in predictions], [1])['per_class'][0]
        identities.append(dict(score, label=name))
    return {'n': len(rows), 'scope': scope, 'identity_per_label': identities,
            'identity_macro_f1': sum(score['f1'] for score in identities) / len(IDENTITIES),
            'target_exact_match': sum(r['target'] == p['target'] for r, p in zip(rows, predictions)) / len(rows)}


### 4.4 子任务模型与 SCL 损失


In [ ]:
class TaskModel(nn.Module):
    def __init__(self, encoder, task, dropout=.1, projection_dim=128):
        super().__init__()
        self.encoder = encoder
        hidden = encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        sizes = {'A': {'a': 2}, 'B': {'b': 3}, 'C': {'scope': 2, 'identity': len(IDENTITIES)}}
        self.heads = nn.ModuleDict({name: nn.Linear(hidden, size) for name, size in sizes[task].items()})
        self.projection = nn.Sequential(nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, projection_dim))

    def forward(self, **inputs):
        sequence = self.encoder(**inputs).last_hidden_state
        mask = inputs['attention_mask'].unsqueeze(-1).to(sequence.dtype)
        pooled = (sequence * mask).sum(1) / mask.sum(1).clamp_min(1)
        hidden = self.dropout(pooled)
        outputs = {name: head(hidden) for name, head in self.heads.items()}
        outputs['projection'] = F.normalize(self.projection(hidden), dim=-1)
        return outputs


In [ ]:
def classification_loss(outputs, batch, task):
    if task == 'A':
        return F.cross_entropy(outputs['a'], batch['stereotype_label'])
    if task == 'B':
        return F.cross_entropy(outputs['b'], batch['hate_label'])
    scope = F.cross_entropy(outputs['scope'], batch['scope_label'])
    identity = F.binary_cross_entropy_with_logits(outputs['identity'], batch['identity_labels'])
    return scope + identity


In [ ]:
def contrastive_labels(batch, task):
    if task == 'A':
        return batch['stereotype_label'].unsqueeze(1)
    if task == 'B':
        return batch['hate_label'].unsqueeze(1)
    # C：对象类型和完整身份标签集合相同，才视为正例。
    return torch.cat([batch['scope_label'].unsqueeze(1), batch['identity_labels'].long()], dim=1)


def contrastive_masks(labels):
    diagonal = torch.eye(len(labels), dtype=torch.bool, device=labels.device)
    positive = (labels[:, None, :] == labels[None, :, :]).all(-1) & ~diagonal
    return positive, ~diagonal


In [ ]:
def contrastive_loss(features, labels, temperature=.1):
    if not math.isfinite(temperature) or temperature <= 0:
        raise ValueError('temperature must be finite and positive')
    # Disable the outer AMP context: .float() alone does not keep matmul in FP32.
    with torch.autocast(device_type=features.device.type, enabled=False):
        positive, allowed = contrastive_masks(labels)
        valid = positive.any(1) & allowed.any(1)
        if not valid.any():
            return features.float().sum()*0
        similarity = F.normalize(features.float(), dim=-1) @ F.normalize(features.float(), dim=-1).T / temperature
        logits = similarity[valid]
        log_denominator = torch.logsumexp(logits.masked_fill(~allowed[valid], float('-inf')), dim=1)
        log_probs = logits-log_denominator[:, None]
        return -(log_probs.masked_fill(~positive[valid], 0).sum(1)/positive[valid].sum(1)).mean()


### 4.5 训练工具与评估


In [ ]:
def write_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


In [ ]:
def move(batch, device):
    return {k: ({ik: iv.to(device) for ik, iv in v.items()} if k == 'inputs'
                else v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}


In [ ]:
@torch.no_grad()
def evaluate(model, loader, config, device):
    model.eval()
    rows, predictions = [], []
    for raw in loader:
        batch = move(raw, device)
        outputs = model(**batch['inputs'])
        predictions.extend(decode(outputs, config['task'], config['identity_threshold']))
        rows.extend(raw['rows'])
    return evaluate_metrics(rows, predictions, config['task'])


In [ ]:
def selection_score(metrics, task):
    return metrics['target_exact_match'] if task == 'C' else metrics['macro_f1']


### 4.6 最佳模型保存


In [ ]:
def save_checkpoint(model, tokenizer, config, root):
    root.mkdir(exist_ok=True)
    model.encoder.config.save_pretrained(root/'encoder_config')
    tokenizer.save_pretrained(root/'tokenizer')
    torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, root/'model.pt')
    write_json(root/'config.json', config)



### 4.7 SCL 训练循环


In [ ]:
def train(config, train_rows, val_rows, run_name):
    config = dict(config)
    root = OUTPUT_DIR / 'model_runs' / run_name
    seed_everything(config['seed'])
    if config['task'] == 'C':
        train_rows = [r for r in train_rows if r['target_loss_mask']]
        val_rows = [r for r in val_rows if r['target_loss_mask']]
    device = torch.device(config["device"])
    root.mkdir(parents=True, exist_ok=True)
    write_json(root/'config.json', config)
    cache = OUTPUT_DIR/'cache'/'huggingface'
    tokenizer = AutoTokenizer.from_pretrained(config['model_name'], cache_dir=cache)
    encoder = AutoModel.from_pretrained(config['model_name'], cache_dir=cache)
    encoder.to(device)
    train_data = TextDataset(train_rows, tokenizer, config)
    val_data = TextDataset(val_rows, tokenizer, config)
    model = TaskModel(encoder, config['task'], config['dropout'], config['projection_dim']).to(device)
    generator = torch.Generator().manual_seed(config['seed'])
    loader = DataLoader(train_data, batch_size=config['batch_size'], shuffle=True, generator=generator,
                        collate_fn=Collator(tokenizer), num_workers=0)
    validation = DataLoader(val_data, batch_size=config['eval_batch_size'], collate_fn=Collator(tokenizer), num_workers=0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])
    steps = math.ceil(len(loader)/config['gradient_accumulation'])*config['epochs']
    scheduler = get_linear_schedule_with_warmup(optimizer, int(steps*config['warmup_ratio']), steps)
    amp_enabled = config.get('mixed_precision', False) and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)
    best, history = -1., []
    for epoch in range(config['epochs']):
        model.train()
        optimizer.zero_grad()
        total, supervised_total, contrast_total = 0., 0., 0.
        for index, raw in enumerate(loader):
            batch = move(raw, device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
                outputs = model(**batch['inputs'])
                supervised = classification_loss(outputs, batch, config['task'])
                features = torch.cat([outputs['projection'], model(**batch['inputs'])['projection']])
                labels = contrastive_labels(batch, config['task']).repeat(2, 1)
                contrast = contrastive_loss(features, labels, config['temperature'])
                loss = supervised + config['contrastive_weight']*contrast
            if not torch.isfinite(loss):
                raise ValueError('训练损失出现非有限数值')
            # 正确缩放最后一个不足完整大小的梯度累积窗口。
            window_start = (index//config['gradient_accumulation'])*config['gradient_accumulation']
            window_size = min(config['gradient_accumulation'], len(loader)-window_start)
            scaler.scale(loss/window_size).backward()
            total += loss.item()
            supervised_total += supervised.item()
            contrast_total += contrast.item()
            if (index+1) % config['gradient_accumulation'] == 0 or index+1 == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                # A reduced scale means AMP skipped optimizer.step() on overflow.
                if scaler.get_scale() >= old_scale:
                    scheduler.step()
                optimizer.zero_grad()
        metrics = evaluate(model, validation, config, device)
        score = selection_score(metrics, config['task'])
        record = {'epoch': epoch+1, 'loss': total/len(loader), 'classification_loss': supervised_total/len(loader),
                  'contrastive_loss': contrast_total/len(loader),
                  'selection_score': score, 'validation': metrics}
        history.append(record)
        write_json(root/'history.json', history)
        metric_name = "对象完全匹配率" if config["task"] == "C" else "Macro-F1"
        print(f"训练轮次：{epoch+1}，总损失：{record['loss']:.6f}，{metric_name}：{score:.6f}", flush=True)
        if score > best:
            best = score
            save_checkpoint(model, tokenizer, config, root/'best')
            write_json(root/'best_metrics.json', metrics)
    return root


## 5. 读取数据划分


In [ ]:
train_rows, validation_rows = load_split(config['data_run'], config['split'])
print("训练样本：", len(train_rows), "验证样本：", len(validation_rows))


## 6. 执行训练


In [ ]:
model_run = train(config, train_rows, validation_rows, RUN_NAME)
print("模型保存目录：", model_run)
